In [0]:
# Import libraries
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql import Row
from datetime import datetime
from pyspark.sql.functions import year, month, col, expr

# Define schema
sales_schema = StructType([
    StructField("OrderID", StringType(), False),
    StructField("OrderDate", DateType(), False),
    StructField("Region", StringType(), True),
    StructField("Product", StringType(), False),
    StructField("Category", StringType(), False),
    StructField("Price", DoubleType(), False),
    StructField("Quantity", IntegerType(), False)
])

# Sample sales data
sales_data = [
    ("ORD009", datetime(2022, 1, 10), "West", "iPhone 15", "Electronics", 1099.99, 3),
    ("ORD010", datetime(2022, 2, 18), "East", "MacBook Pro", "Electronics", 1999.99, 1),
    ("ORD011", datetime(2022, 3, 6), "South", "Ergonomic Chair", "Furniture", 179.99, 2),
    ("ORD012", datetime(2022, 4, 20), "North", "4K Monitor", "Electronics", 329.99, 2),
    ("ORD013", datetime(2022, 5, 27), "West", "Standing Desk", "Furniture", 499.99, 1),
    ("ORD014", datetime(2022, 6, 15), "East", "iPad Pro", "Electronics", 899.99, 2),
    ("ORD015", datetime(2022, 7, 23), "South", "Corner Bookshelf", "Furniture", 249.99, 3),
    ("ORD016", datetime(2022, 8, 12), "North", "Laser Printer", "Electronics", 199.99, 1),
    ("ORD001", datetime(2023, 1, 15), "West", "iPhone 14", "Electronics", 999.99, 2),
    ("ORD002", datetime(2023, 2, 20), "East", "MacBook Air", "Electronics", 1199.49, 1),
    ("ORD003", datetime(2023, 3, 5), "South", "Desk Chair", "Furniture", 149.99, 4),
    ("ORD004", datetime(2023, 4, 17), "North", "Monitor", "Electronics", 249.99, 2),
    ("ORD005", datetime(2023, 5, 30), "West", "Office Desk", "Furniture", 399.99, 1),
    ("ORD006", datetime(2023, 6, 12), "East", "iPad", "Electronics", 499.99, 3),
    ("ORD007", datetime(2023, 7, 25), "South", "Bookshelf", "Furniture", 199.99, 2),
    ("ORD008", datetime(2023, 8, 8), "North", "Printer", "Electronics", 149.49, 1),
    ("ORD009", datetime(2024, 1, 10), "West", "iPhone 15", "Electronics", 1099.99, 3),
    ("ORD010", datetime(2024, 2, 18), "East", "MacBook Pro", "Electronics", 1999.99, 1),
    ("ORD011", datetime(2024, 3, 6), "South", "Ergonomic Chair", "Furniture", 179.99, 2),
    ("ORD012", datetime(2024, 4, 20), "North", "4K Monitor", "Electronics", 329.99, 2),
    ("ORD013", datetime(2024, 5, 27), "West", "Standing Desk", "Furniture", 499.99, 1),
    ("ORD014", datetime(2024, 6, 15), "East", "iPad Pro", "Electronics", 899.99, 2),
    ("ORD015", datetime(2024, 7, 23), "South", "Corner Bookshelf", "Furniture", 249.99, 3),
    ("ORD016", datetime(2024, 8, 12), "North", "Laser Printer", "Electronics", 199.99, 1),
    ("ORD009", datetime(2025, 1, 10), "West", "iPhone 15", "Electronics", 1099.99, 3),
	("ORD010", datetime(2025, 2, 18), "East", "MacBook Pro", "Electronics", 1999.99, 1),
	("ORD011", datetime(2025, 3, 6), "South", "Ergonomic Chair", "Furniture", 179.99, 2),
	("ORD012", datetime(2025, 4, 20), "North", "4K Monitor", "Electronics", 329.99, 2),
	("ORD013", datetime(2025, 5, 27), "West", "Standing Desk", "Furniture", 499.99, 1),
	("ORD014", datetime(2025, 6, 15), "East", "iPad Pro", "Electronics", 899.99, 2),
	("ORD015", datetime(2025, 7, 23), "South", "Corner Bookshelf", "Furniture", 249.99, 3),
	("ORD016", datetime(2025, 8, 12), "North", "Laser Printer", "Electronics", 199.99, 1)

]

# Create DataFrame
sales_df = spark.createDataFrame(sales_data, schema=sales_schema)

# Basic assertions on raw data

# Check if dataframe is empty
assert sales_df.count() > 0, "❌ DataFrame is empty!"
# Check if schema has all 7 columns
assert len(sales_df.columns) == 7, "❌ Schema mismatch: Expected 7 columns."
# Check if orderId has null values
assert sales_df.filter(col("OrderID").isNull()).count() == 0, "❌ OrderID contains null values."

# Check if price or quantity has any negative values
assert sales_df.filter(col("Price") <= 0).count() == 0, "❌ Price must be greater than 0."
assert sales_df.filter(col("Quantity") <= 0).count() == 0, "❌ Quantity must be greater than 0."

In [0]:
# Transformations
sales_transformed = (sales_df
    .withColumn("Year", year(col("OrderDate")))
    .withColumn("Month", month(col("OrderDate")))
    .withColumn("SalesAmount", expr("Price * Quantity")))

# Assertions on transformed data

# Check if SalesAmount column exists
assert "SalesAmount" in sales_transformed.columns, "❌ SalesAmount column missing!"

# Check if SalesAmount has any negative values
assert sales_transformed.filter(col("SalesAmount") <= 0).count() == 0, "❌ SalesAmount must be > 0."

# Check if Year column is populated
assert sales_transformed.select("Year").distinct().count() > 0, "❌ Year column not populated."    
     

In [0]:
# Aggregations
sales_summary = (sales_transformed
    .groupBy("Region", "Category")
    .sum("SalesAmount")
    .withColumnRenamed("sum(SalesAmount)", "TotalSales")
    .orderBy("Region", "Category"))

sales_summary.display()

# Assertions on summary

# Check if aggregation produces rows
assert sales_summary.count() > 0, "❌ Aggregation produced no results!"
# Check if totalSales column exists
assert "TotalSales" in sales_summary.columns, "❌ TotalSales column missing!"

Region,Category,TotalSales
East,Electronics,14099.369999999999
North,Electronics,3229.38
South,Furniture,4329.79
West,Electronics,11899.890000000003
West,Furniture,1899.96


In [0]:
# Save as Delta Table (overwrite mode)
sales_transformed.write.format("delta").mode("overwrite").saveAsTable("`ci-cd_databricks_pipeline`.sales_schema.sales_git_table")

# If all tests pass
print("✅ All tests passed successfully!")

✅ All tests passed successfully!
